# Heart Failure Associations from HERMES GWAS

This notebook extracts heart failure GWAS associations from the HERMES summary statistics file for all gene regions identified in the Ensembl gene-region step.

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
gene_regions_file = Path("../data/interim/ensembl/gene_regions_grch37.csv")

hermes_path = Path("../data/raw/hermes/hermes_hf.tsv")

out_dir = Path("../data/interim/hermes")
out_dir.mkdir(parents=True, exist_ok=True)

out_associations_csv = out_dir / "associations.csv"

In [3]:
gene_regions = pd.read_csv(gene_regions_file)

gene_regions.head()

,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,45409011,45412650,1,+,GRCh37,50000,45359011,45462650,45359011,45409010,45412651,45462650,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,45003675,45011075,1,+,GRCh37,50000,44953675,45061075,44953675,45003674,45011076,45061075,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22962999,22966101,1,+,GRCh37,50000,22912999,23016101,22912999,22962998,22966102,23016101,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22979255,22988031,1,+,GRCh37,50000,22929255,23038031,22929255,22979254,22988032,23038031,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22970123,22974603,1,+,GRCh37,50000,22920123,23024603,22920123,22970122,22974604,23024603,protein_coding,Gene,Ensembl REST


In [4]:
genes_for_hermes = gene_regions.loc[
    gene_regions["lookup_status"] == "found"
].copy()

genes_for_hermes = (
    genes_for_hermes
    .dropna(subset=["official_gene_symbol", "chromosome", "region_start", "region_end"])
    .drop_duplicates(subset=["official_gene_symbol"])
    .reset_index(drop=True)
)

print("Genes available for HERMES filtering:", len(genes_for_hermes))

genes_for_hermes.head()

Genes available for HERMES filtering: 51


,input_gene_symbol,official_gene_symbol,ensembl_gene_id,lookup_status,species,chromosome,gene_start,gene_end,strand,strand_symbol,assembly_name,flanking_bp,region_start,region_end,upstream_start,upstream_end,downstream_start,downstream_end,biotype,object_type,source
0,APOE,APOE,ENSG00000130203,found,homo_sapiens,19,45409011,45412650,1,+,GRCh37,50000,45359011,45462650,45359011,45409010,45412651,45462650,protein_coding,Gene,Ensembl REST
1,B2M,B2M,ENSG00000166710,found,homo_sapiens,15,45003675,45011075,1,+,GRCh37,50000,44953675,45061075,44953675,45003674,45011076,45061075,protein_coding,Gene,Ensembl REST
2,C1QA,C1QA,ENSG00000173372,found,homo_sapiens,1,22962999,22966101,1,+,GRCh37,50000,22912999,23016101,22912999,22962998,22966102,23016101,protein_coding,Gene,Ensembl REST
3,C1QB,C1QB,ENSG00000173369,found,homo_sapiens,1,22979255,22988031,1,+,GRCh37,50000,22929255,23038031,22929255,22979254,22988032,23038031,protein_coding,Gene,Ensembl REST
4,C1QC,C1QC,ENSG00000159189,found,homo_sapiens,1,22970123,22974603,1,+,GRCh37,50000,22920123,23024603,22920123,22970122,22974604,23024603,protein_coding,Gene,Ensembl REST


In [5]:
if not hermes_path.exists():
    raise FileNotFoundError(
        f"HERMES file not found at: {hermes_path}\n"
    )

In [6]:
print("HERMES filtering setup")
print("Assembly:", genes_for_hermes["assembly_name"].dropna().unique())
print("Number of gene regions:", len(genes_for_hermes))

genes_for_hermes[
    [
        "official_gene_symbol",
        "chromosome",
        "gene_start",
        "gene_end",
        "region_start",
        "region_end",
        "strand_symbol",
        "assembly_name"
    ]
].head()

HERMES filtering setup
Assembly: ['GRCh37']
Number of gene regions: 51


,official_gene_symbol,chromosome,gene_start,gene_end,region_start,region_end,strand_symbol,assembly_name
0,APOE,19,45409011,45412650,45359011,45462650,+,GRCh37
1,B2M,15,45003675,45011075,44953675,45061075,+,GRCh37
2,C1QA,1,22962999,22966101,22912999,23016101,+,GRCh37
3,C1QB,1,22979255,22988031,22929255,23038031,+,GRCh37
4,C1QC,1,22970123,22974603,22920123,23024603,+,GRCh37


In [7]:
def annotate_location_relative_to_gene(position, gene_start, gene_end, strand):
    """
    Annotate whether a variant is inside the gene body,
    upstream, or downstream, taking gene strand into account.
    """
    if pd.isna(position) or pd.isna(gene_start) or pd.isna(gene_end):
        return None

    position = int(position)
    gene_start = int(gene_start)
    gene_end = int(gene_end)

    if gene_start <= position <= gene_end:
        return "intragenic"

    if str(strand) in ["1", "+", "+1"]:
        if position < gene_start:
            return "upstream"
        return "downstream"

    if str(strand) in ["-1", "-", "−"]:
        if position > gene_end:
            return "upstream"
        return "downstream"

    return None


def calculate_distance_from_gene(position, gene_start, gene_end):
    """
    Calculate distance from the gene body.
    Variants inside the gene body have distance 0.
    """
    if pd.isna(position) or pd.isna(gene_start) or pd.isna(gene_end):
        return None

    position = int(position)
    gene_start = int(gene_start)
    gene_end = int(gene_end)

    if gene_start <= position <= gene_end:
        return 0

    if position < gene_start:
        return gene_start - position

    return position - gene_end

In [8]:
usecols = [
    "MarkerName",
    "Allele1",
    "Allele2",
    "Freq1",
    "FreqSE",
    "MinFreq",
    "MaxFreq",
    "Effect",
    "StdErr",
    "P.value",
    "Direction",
    "HetISq",
    "HetChiSq",
    "HetDf",
    "HetPVal",
    "TotalSampleSize",
    "nstudies",
    "final_rsid",
    "CHR",
    "BP"
]

In [9]:
matched_chunks = []

regions_for_filtering = genes_for_hermes.copy()

regions_for_filtering["chromosome_filter"] = (
    regions_for_filtering["chromosome"]
    .astype(str)
    .str.replace("chr", "", regex=False)
)

regions_for_filtering["region_start"] = pd.to_numeric(
    regions_for_filtering["region_start"],
    errors="coerce"
)

regions_for_filtering["region_end"] = pd.to_numeric(
    regions_for_filtering["region_end"],
    errors="coerce"
)

for chunk in pd.read_csv(
    hermes_path,
    sep="\t",
    usecols=usecols,
    chunksize=500_000
):
    chunk["CHR_filter"] = (
        chunk["CHR"]
        .astype(str)
        .str.replace("chr", "", regex=False)
    )

    chunk["BP_filter"] = pd.to_numeric(chunk["BP"], errors="coerce")

    for _, gene_region in regions_for_filtering.iterrows():
        region_chromosome = gene_region["chromosome_filter"]
        region_start = int(gene_region["region_start"])
        region_end = int(gene_region["region_end"])

        matched = chunk[
            (chunk["CHR_filter"] == region_chromosome) &
            (chunk["BP_filter"].between(region_start, region_end))
        ].copy()

        if not matched.empty:
            matched["query_source"] = "gene_region"
            matched["query_value"] = gene_region["official_gene_symbol"]
            matched["region_assembly"] = gene_region["assembly_name"]
            matched["gene_start"] = int(gene_region["gene_start"])
            matched["gene_end"] = int(gene_region["gene_end"])
            matched["region_start"] = region_start
            matched["region_end"] = region_end
            matched["strand"] = gene_region["strand"]
            matched["strand_symbol"] = gene_region["strand_symbol"]

            matched["location_relative_to_gene"] = matched["BP_filter"].apply(
                lambda position: annotate_location_relative_to_gene(
                    position,
                    gene_region["gene_start"],
                    gene_region["gene_end"],
                    gene_region["strand"]
                )
            )

            matched["distance_from_gene"] = matched["BP_filter"].apply(
                lambda position: calculate_distance_from_gene(
                    position,
                    gene_region["gene_start"],
                    gene_region["gene_end"]
                )
            )

            matched = matched.drop(columns=["CHR_filter", "BP_filter"])

            matched_chunks.append(matched)

if matched_chunks:
    hermes_matches_df = pd.concat(matched_chunks, ignore_index=True)
else:
    hermes_matches_df = pd.DataFrame(columns=usecols)

hermes_matches_df.shape

(21060, 31)

In [10]:
hermes_associations_df = hermes_matches_df.rename(columns={
    "final_rsid": "rsID",
    "MarkerName": "HERMES_marker_name",
    "CHR": "HERMES_CHR",
    "BP": "HERMES_BP",
    "Allele1": "HERMES_Allele1",
    "Allele2": "HERMES_Allele2",
    "Freq1": "HERMES_Allele1_freq",
    "FreqSE": "HERMES_freq_se",
    "MinFreq": "HERMES_min_freq",
    "MaxFreq": "HERMES_max_freq",
    "Effect": "HERMES_effect",
    "StdErr": "HERMES_std_error",
    "P.value": "HERMES_p_value",
    "Direction": "HERMES_direction",
    "HetISq": "HERMES_het_i_sq",
    "HetChiSq": "HERMES_het_chi_sq",
    "HetDf": "HERMES_het_df",
    "HetPVal": "HERMES_het_p_value",
    "TotalSampleSize": "HERMES_total_sample_size",
    "nstudies": "HERMES_n_studies"
})

hermes_associations_df.head()

,HERMES_marker_name,HERMES_Allele1,HERMES_Allele2,HERMES_Allele1_freq,HERMES_freq_se,HERMES_min_freq,HERMES_max_freq,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_direction,HERMES_het_i_sq,HERMES_het_chi_sq,HERMES_het_df,HERMES_het_p_value,HERMES_total_sample_size,HERMES_n_studies,rsID,HERMES_CHR,HERMES_BP,query_source,query_value,region_assembly,gene_start,gene_end,region_start,region_end,strand,strand_symbol,location_relative_to_gene,distance_from_gene
0,1:22913153,t,c,0.6525,0.0225,0.6284,0.7078,0.0161,0.0092,0.08058,---+++-++-+---+++-?+-----+++,0.0,23.094,26,0.62760,561521.0,27,rs638058,1,22913153,gene_region,C1QA,GRCh37,22962999,22966101,22912999,23016101,1,+,upstream,49846
1,1:22913606,a,g,0.1364,0.0092,0.1119,0.1457,-0.0142,0.0129,0.26950,+-----++-+-+++-+++?--++++---,36.5,40.952,26,0.03138,561524.0,27,rs12066127,1,22913606,gene_region,C1QA,GRCh37,22962999,22966101,22912999,23016101,1,+,upstream,49393
2,1:22913920,t,g,0.1364,0.0093,0.1112,0.1467,-0.0140,0.0129,0.27850,+-----++-+-+++-+++?--++++---,35.8,40.497,26,0.03484,561521.0,27,rs10917261,1,22913920,gene_region,C1QA,GRCh37,22962999,22966101,22912999,23016101,1,+,upstream,49079
3,1:22913983,a,g,0.1529,0.0124,0.1274,0.1686,-0.0104,0.0123,0.39750,+-+---++-+-+++-+++?--++++---,39.6,43.047,26,0.01909,561516.0,27,rs10917262,1,22913983,gene_region,C1QA,GRCh37,22962999,22966101,22912999,23016101,1,+,upstream,49016
4,1:22913993,a,g,0.6517,0.0229,0.6263,0.7081,0.0190,0.0094,0.04446,-?-+++-++-+-??+++-?+-----+++,0.0,19.353,23,0.68060,554609.0,24,rs624005,1,22913993,gene_region,C1QA,GRCh37,22962999,22966101,22912999,23016101,1,+,upstream,49006


In [11]:
numeric_cols = [
    "HERMES_CHR",
    "HERMES_BP",
    "HERMES_Allele1_freq",
    "HERMES_freq_se",
    "HERMES_min_freq",
    "HERMES_max_freq",
    "HERMES_effect",
    "HERMES_std_error",
    "HERMES_p_value",
    "HERMES_het_i_sq",
    "HERMES_het_chi_sq",
    "HERMES_het_df",
    "HERMES_het_p_value",
    "HERMES_total_sample_size",
    "HERMES_n_studies",
    "gene_start",
    "gene_end",
    "region_start",
    "region_end",
    "distance_from_gene"
]

for col in numeric_cols:
    if col in hermes_associations_df.columns:
        hermes_associations_df[col] = pd.to_numeric(
            hermes_associations_df[col],
            errors="coerce"
        )

hermes_associations_df = hermes_associations_df.sort_values(
    ["query_value", "HERMES_p_value"],
    ascending=[True, True],
    na_position="last"
).reset_index(drop=True)

hermes_associations_df.head()

,HERMES_marker_name,HERMES_Allele1,HERMES_Allele2,HERMES_Allele1_freq,HERMES_freq_se,HERMES_min_freq,HERMES_max_freq,HERMES_effect,HERMES_std_error,HERMES_p_value,HERMES_direction,HERMES_het_i_sq,HERMES_het_chi_sq,HERMES_het_df,HERMES_het_p_value,HERMES_total_sample_size,HERMES_n_studies,rsID,HERMES_CHR,HERMES_BP,query_source,query_value,region_assembly,gene_start,gene_end,region_start,region_end,strand,strand_symbol,location_relative_to_gene,distance_from_gene
0,19:45418790,t,c,0.7741,0.0115,0.7454,0.8100,0.0405,0.0113,0.000318,++--??+--?--++-+++?++-++++-+,31.0,33.351,23,0.075140,539515.0,24,rs5117,19,45418790,gene_region,APOE,GRCh37,45409011,45412650,45359011,45462650,1,+,downstream,6140
1,19:45416741,t,c,0.2306,0.0122,0.2070,0.2610,-0.0377,0.0108,0.000459,+-++?+-++?++--+---?--+----+-,52.0,50.033,24,0.001402,558838.0,25,rs438811,19,45416741,gene_region,APOE,GRCh37,45409011,45412650,45359011,45462650,1,+,downstream,4091
2,19:45415713,a,g,0.1332,0.0191,0.0959,0.1681,-0.0471,0.0140,0.000748,--+-??-+-?-+----++?--+----+-,0.0,17.499,23,0.784100,539527.0,24,rs10414043,19,45415713,gene_region,APOE,GRCh37,45409011,45412650,45359011,45462650,1,+,downstream,3063
3,19:45411941,t,c,0.8426,0.0213,0.8119,0.8855,0.0414,0.0126,0.001003,++-+?-+--?--++++-+?++--+++-+,29.8,34.202,24,0.081140,558869.0,25,rs429358,19,45411941,gene_region,APOE,GRCh37,45409011,45412650,45359011,45462650,1,+,intragenic,0
4,19:45415935,t,g,0.1330,0.0197,0.0949,0.1706,-0.0442,0.0141,0.001709,--+-??-+-?++----?+?--+--?-+-,0.0,12.254,21,0.932500,537451.0,22,rs7256200,19,45415935,gene_region,APOE,GRCh37,45409011,45412650,45359011,45462650,1,+,downstream,3285


In [12]:
rows_before_deduplication = len(hermes_associations_df)

hermes_associations_df = hermes_associations_df.drop_duplicates()

rows_after_deduplication = len(hermes_associations_df)
duplicate_rows_removed = rows_before_deduplication - rows_after_deduplication

hermes_associations_df.to_csv(out_associations_csv, index=False)

duplicate_rows_removed

0

In [13]:
summary = {
    "phenotype": "Heart failure",
    "genes_available_for_filtering": int(len(genes_for_hermes)),
    "genes_with_hermes_associations": int(
        hermes_associations_df["query_value"].nunique()
    ) if not hermes_associations_df.empty else 0,
    "region_assembly": "GRCh37",
    "associations": int(len(hermes_associations_df)),
    "unique_rsIDs": int(hermes_associations_df["rsID"].nunique()),
    "duplicate_rows_removed": int(duplicate_rows_removed),
}

summary

{'phenotype': 'Heart failure',
 'genes_available_for_filtering': 51,
 'genes_with_hermes_associations': 50,
 'region_assembly': 'GRCh37',
 'associations': 21060,
 'unique_rsIDs': 20084,
 'duplicate_rows_removed': 0}